In [1]:
import os
import shutil
os.makedirs('data', exist_ok=True)

source_path = '/content/synthetic_survey_dataset.csv'
destination_path = 'data/synthetic_survey_dataset.csv'
shutil.move(source_path, destination_path)

print(f"Moved '{source_path}' to '{destination_path}'")

Moved '/content/synthetic_survey_dataset.csv' to 'data/synthetic_survey_dataset.csv'


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd


DATA_PATH = Path("data/synthetic_survey_dataset.csv")
RESULTS_DIR = Path("results")

TOTAL_SAMPLE = 300

TEAM_CAPACITY = {
    "Team_A": 85,
    "Team_B": 80,
    "Team_C": 75,
    "Team_D": 70,
}

# Relative efficiency of each team in each type of area.
# Values below 1 mean that the team is relatively cheaper there.
TEAM_FACTOR = {
    "Team_A": {
        "Urban": 0.90,
        "Semi-Urban": 1.00,
        "Rural": 1.15,
    },
    "Team_B": {
        "Urban": 1.00,
        "Semi-Urban": 0.92,
        "Rural": 1.10,
    },
    "Team_C": {
        "Urban": 1.08,
        "Semi-Urban": 0.95,
        "Rural": 0.90,
    },
    "Team_D": {
        "Urban": 1.05,
        "Semi-Urban": 1.00,
        "Rural": 0.88,
    },
}


def largest_remainder(values, total):
    """
    Convert fractional allocations to integers while preserving the total.
    """
    base = np.floor(values).astype(int)
    remainder = int(total - base.sum())

    fractions = values - base
    order = np.argsort(-fractions)

    for i in order[:remainder]:
        base[i] += 1

    return base


def neyman_allocation(df):
    summary = (
        df.groupby("Stratum")
        .agg(
            Population=("Respondent_ID", "count"),
            Mean_Income=("Monthly_Income", "mean"),
            Std_Income=("Monthly_Income", "std"),
        )
        .reset_index()
    )

    summary["N_times_S"] = (
        summary["Population"] * summary["Std_Income"]
    )

    raw = (
        TOTAL_SAMPLE
        * summary["N_times_S"]
        / summary["N_times_S"].sum()
    )

    summary["Neyman_Sample"] = largest_remainder(
        raw.to_numpy(),
        TOTAL_SAMPLE,
    )

    return summary


def stratum_costs(df):
    """
    Estimate a baseline interview cost for each stratum from the
    respondent-level survey data.
    """
    summary = (
        df.groupby("Stratum")
        .agg(
            Avg_Interview_Cost=("Interview_Cost", "mean"),
            Avg_Distance=("GPS_Distance_km", "mean"),
            Avg_Interview_Time=("Survey_Time_Min", "mean"),
            Avg_Travel_Difficulty=("Travel_Difficulty", "mean"),
        )
        .reset_index()
    )

    # Interview_Cost already contains the main cost signal.
    # The other terms represent additional fieldwork burden.
    summary["Base_Cost"] = (
        summary["Avg_Interview_Cost"]
        + 0.10 * summary["Avg_Distance"]
        + 0.03 * summary["Avg_Interview_Time"]
        + 0.25 * summary["Avg_Travel_Difficulty"]
    )

    return summary


def build_cost_matrix(base_cost):
    rows = []

    base = dict(
        zip(
            base_cost["Stratum"],
            base_cost["Base_Cost"],
        )
    )

    for team, factors in TEAM_FACTOR.items():
        for stratum, factor in factors.items():
            rows.append(
                {
                    "Team": team,
                    "Stratum": stratum,
                    "Cost": round(base[stratum] * factor, 2),
                }
            )

    return pd.DataFrame(rows)


def main():
    RESULTS_DIR.mkdir(exist_ok=True)

    df = pd.read_csv(DATA_PATH)

    required = {
        "Respondent_ID",
        "Stratum",
        "Monthly_Income",
        "Interview_Cost",
        "GPS_Distance_km",
        "Survey_Time_Min",
        "Travel_Difficulty",
    }

    missing = required.difference(df.columns)

    if missing:
        raise ValueError(
            f"Dataset is missing columns: {sorted(missing)}"
        )

    neyman = neyman_allocation(df)
    baseline = stratum_costs(df)
    costs = build_cost_matrix(baseline)

    capacities = pd.DataFrame(
        {
            "Team": TEAM_CAPACITY.keys(),
            "Capacity": TEAM_CAPACITY.values(),
        }
    )

    if capacities["Capacity"].sum() < TOTAL_SAMPLE:
        raise ValueError(
            "Total team capacity is smaller than the required sample."
        )

    neyman.to_csv(
        RESULTS_DIR / "neyman_allocation.csv",
        index=False,
    )

    costs.to_csv(
        RESULTS_DIR / "cost_matrix.csv",
        index=False,
    )

    capacities.to_csv(
        RESULTS_DIR / "team_capacity.csv",
        index=False,
    )

    print("\nNeyman allocation")
    print(
        neyman[
            [
                "Stratum",
                "Population",
                "Std_Income",
                "Neyman_Sample",
            ]
        ].to_string(index=False)
    )

    print("\nTeam-stratum cost matrix")
    print(
        costs.pivot(
            index="Team",
            columns="Stratum",
            values="Cost",
        ).round(2)
    )

    print("\nTeam capacities")
    print(capacities.to_string(index=False))


if __name__ == "__main__":
    main()


Neyman allocation
   Stratum  Population   Std_Income  Neyman_Sample
     Rural         350 10476.910948             59
Semi-Urban         230 15162.809669             56
     Urban         420 27342.851593            185

Team-stratum cost matrix
Stratum  Rural  Semi-Urban  Urban
Team                             
Team_A   28.93       22.02  18.11
Team_B   27.67       20.26  20.12
Team_C   22.64       20.92  21.73
Team_D   22.14       22.02  21.13

Team capacities
  Team  Capacity
Team_A        85
Team_B        80
Team_C        75
Team_D        70


In [3]:
from pathlib import Path

import pandas as pd
import pulp


RESULTS_DIR = Path("results")


def load_inputs():
    costs = pd.read_csv(RESULTS_DIR / "cost_matrix.csv")
    capacities = pd.read_csv(
        RESULTS_DIR / "team_capacity.csv"
    )
    targets = pd.read_csv(
        RESULTS_DIR / "neyman_allocation.csv"
    )

    return costs, capacities, targets


def solve_model(costs, capacities, targets):
    teams = capacities["Team"].tolist()
    strata = targets["Stratum"].tolist()

    capacity = capacities.set_index("Team")[
        "Capacity"
    ].to_dict()

    demand = targets.set_index("Stratum")[
        "Neyman_Sample"
    ].to_dict()

    cost = {
        (row.Team, row.Stratum): row.Cost
        for row in costs.itertuples()
    }

    if sum(capacity.values()) < sum(demand.values()):
        raise ValueError(
            "The model is infeasible: total capacity "
            "is smaller than total demand."
        )

    model = pulp.LpProblem(
        "Survey_Fieldwork_Allocation",
        pulp.LpMinimize,
    )

    x = pulp.LpVariable.dicts(
        "interviews",
        (teams, strata),
        lowBound=0,
        cat=pulp.LpInteger,
    )

    model += pulp.lpSum(
        cost[t, s] * x[t][s]
        for t in teams
        for s in strata
    ), "Total_Fieldwork_Cost"

    for s in strata:
        model += (
            pulp.lpSum(x[t][s] for t in teams)
            == demand[s],
            f"Target_{s}",
        )

    for t in teams:
        model += (
            pulp.lpSum(x[t][s] for s in strata)
            <= capacity[t],
            f"Capacity_{t}",
        )

    solver = pulp.PULP_CBC_CMD(msg=False)
    model.solve(solver)

    if pulp.LpStatus[model.status] != "Optimal":
        raise RuntimeError(
            "No optimal solution found. "
            f"Solver status: {pulp.LpStatus[model.status]}"
        )

    rows = []

    for t in teams:
        for s in strata:
            allocation = int(round(pulp.value(x[t][s])))

            rows.append(
                {
                    "Team": t,
                    "Stratum": s,
                    "Allocation": allocation,
                    "Unit_Cost": cost[t, s],
                    "Total_Cost": allocation * cost[t, s],
                }
            )

    solution = pd.DataFrame(rows)

    return model, solution


def main():
    costs, capacities, targets = load_inputs()

    model, solution = solve_model(
        costs,
        capacities,
        targets,
    )

    solution.to_csv(
        RESULTS_DIR / "optimal_allocation.csv",
        index=False,
    )

    total_cost = solution["Total_Cost"].sum()

    print("Solver status: Optimal")
    print(f"Minimum fieldwork cost: {total_cost:,.2f}")

    print("\nOptimal allocation")
    print(
        solution.pivot(
            index="Team",
            columns="Stratum",
            values="Allocation",
        ).fillna(0).astype(int)
    )

    utilization = (
        solution.groupby("Team")["Allocation"]
        .sum()
        .rename("Used")
        .reset_index()
        .merge(capacities, on="Team")
    )

    utilization["Utilization_Percent"] = (
        100 * utilization["Used"] / utilization["Capacity"]
    )

    print("\nTeam utilization")
    print(utilization.to_string(index=False))


if __name__ == "__main__":
    main()

Solver status: Optimal
Minimum fieldwork cost: 6,053.83

Optimal allocation
Stratum  Rural  Semi-Urban  Urban
Team                             
Team_A       0           0     85
Team_B       0           0     80
Team_C       9          56      0
Team_D      50           0     20

Team utilization
  Team  Used  Capacity  Utilization_Percent
Team_A    85        85           100.000000
Team_B    80        80           100.000000
Team_C    65        75            86.666667
Team_D    70        70           100.000000


In [4]:
from pathlib import Path

import pandas as pd
from pyomo.environ import (
    ConcreteModel,
    Constraint,
    NonNegativeIntegers,
    Objective,
    Set,
    SolverFactory,
    Var,
    minimize,
    value,
)
from pyomo.opt import SolverStatus, TerminationCondition


RESULTS_DIR = Path("results")


def main():
    costs = pd.read_csv(
        RESULTS_DIR / "cost_matrix.csv"
    )

    capacities = pd.read_csv(
        RESULTS_DIR / "team_capacity.csv"
    )

    targets = pd.read_csv(
        RESULTS_DIR / "neyman_allocation.csv"
    )

    teams = capacities["Team"].tolist()
    strata = targets["Stratum"].tolist()

    capacity = capacities.set_index("Team")[
        "Capacity"
    ].to_dict()

    demand = targets.set_index("Stratum")[
        "Neyman_Sample"
    ].to_dict()

    unit_cost = {
        (row.Team, row.Stratum): row.Cost
        for row in costs.itertuples()
    }

    model = ConcreteModel()

    model.TEAMS = Set(initialize=teams)
    model.STRATA = Set(initialize=strata)

    model.x = Var(
        model.TEAMS,
        model.STRATA,
        domain=NonNegativeIntegers,
    )

    model.total_cost = Objective(
        expr=sum(
            unit_cost[t, s] * model.x[t, s]
            for t in model.TEAMS
            for s in model.STRATA
        ),
        sense=minimize,
    )

    def target_rule(m, s):
        return (
            sum(m.x[t, s] for t in m.TEAMS)
            == demand[s]
        )

    model.target = Constraint(
        model.STRATA,
        rule=target_rule,
    )

    def capacity_rule(m, t):
        return (
            sum(m.x[t, s] for s in m.STRATA)
            <= capacity[t]
        )

    model.team_capacity = Constraint(
        model.TEAMS,
        rule=capacity_rule,
    )


    solver = SolverFactory("appsi_highs")

    result = solver.solve(model)

    if (
        result.solver.status != SolverStatus.ok
        or result.solver.termination_condition
        != TerminationCondition.optimal
    ):
        raise RuntimeError(
            "Pyomo did not obtain an optimal solution."
        )

    rows = []

    for t in teams:
        for s in strata:
            allocation = int(round(value(model.x[t, s])))

            rows.append(
                {
                    "Team": t,
                    "Stratum": s,
                    "Allocation": allocation,
                    "Unit_Cost": unit_cost[t, s],
                    "Total_Cost": allocation * unit_cost[t, s],
                }
            )

    solution = pd.DataFrame(rows)

    solution.to_csv(
        RESULTS_DIR / "pyomo_allocation.csv",
        index=False,
    )

    print(
        f"Pyomo objective value: "
        f"{value(model.total_cost):,.2f}"
    )

    print(
        solution.pivot(
            index="Team",
            columns="Stratum",
            values="Allocation",
        ).fillna(0).astype(int)
    )


if __name__ == "__main__":
    main()

Pyomo objective value: 6,053.83
Stratum  Rural  Semi-Urban  Urban
Team                             
Team_A       0           0     85
Team_B       0           0     80
Team_C       9          56      0
Team_D      50           0     20


In [5]:
from pathlib import Path

import pandas as pd


RESULTS_DIR = Path("results")

pulp_result = pd.read_csv(
    RESULTS_DIR / "optimal_allocation.csv"
)

pyomo_result = pd.read_csv(
    RESULTS_DIR / "pyomo_allocation.csv"
)

pulp_cost = pulp_result["Total_Cost"].sum()
pyomo_cost = pyomo_result["Total_Cost"].sum()

print(f"PuLP objective : {pulp_cost:.2f}")
print(f"Pyomo objective: {pyomo_cost:.2f}")

if abs(pulp_cost - pyomo_cost) < 1e-6:
    print("Both implementations give the same optimal cost.")
else:
    print("Warning: solver objective values differ.")

PuLP objective : 6053.83
Pyomo objective: 6053.83
Both implementations give the same optimal cost.


In [6]:
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd


RESULTS_DIR = Path("results")
FIGURES_DIR = Path("figures")

FIGURES_DIR.mkdir(exist_ok=True)


def plot_neyman():
    data = pd.read_csv(
        RESULTS_DIR / "neyman_allocation.csv"
    )

    fig, ax = plt.subplots(figsize=(7, 4.5))

    ax.bar(
        data["Stratum"],
        data["Neyman_Sample"],
    )

    ax.set_title("Neyman Allocation by Stratum")
    ax.set_xlabel("Stratum")
    ax.set_ylabel("Required Sample")

    for i, value in enumerate(data["Neyman_Sample"]):
        ax.text(
            i,
            value + 3,
            str(value),
            ha="center",
        )

    fig.tight_layout()

    fig.savefig(
        FIGURES_DIR / "neyman_allocation.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)


def plot_allocation():
    allocation = pd.read_csv(
        RESULTS_DIR / "optimal_allocation.csv"
    )

    pivot = allocation.pivot(
        index="Stratum",
        columns="Team",
        values="Allocation",
    )

    ax = pivot.plot(
        kind="bar",
        figsize=(9, 5),
    )

    ax.set_title("Optimal Field-Team Allocation")
    ax.set_xlabel("Stratum")
    ax.set_ylabel("Number of Interviews")
    ax.legend(title="Team")

    plt.xticks(rotation=0)
    plt.tight_layout()

    plt.savefig(
        FIGURES_DIR / "optimal_allocation.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.close()


def plot_network():
    allocation = pd.read_csv(
        RESULTS_DIR / "optimal_allocation.csv"
    )

    capacities = pd.read_csv(
        RESULTS_DIR / "team_capacity.csv"
    )

    targets = pd.read_csv(
        RESULTS_DIR / "neyman_allocation.csv"
    )

    G = nx.DiGraph()

    source = "Source"
    sink = "Sink"

    teams = capacities["Team"].tolist()
    strata = targets["Stratum"].tolist()

    for row in capacities.itertuples():
        G.add_edge(
            source,
            row.Team,
            flow=int(
                allocation.loc[
                    allocation["Team"] == row.Team,
                    "Allocation",
                ].sum()
            ),
        )

    for row in allocation.itertuples():
        if row.Allocation > 0:
            G.add_edge(
                row.Team,
                row.Stratum,
                flow=int(row.Allocation),
            )

    for row in targets.itertuples():
        G.add_edge(
            row.Stratum,
            sink,
            flow=int(row.Neyman_Sample),
        )

    pos = {
        source: (0, 1.5),
        "Team_A": (1, 3),
        "Team_B": (1, 2),
        "Team_C": (1, 1),
        "Team_D": (1, 0),
        "Urban": (2, 2.5),
        "Semi-Urban": (2, 1.5),
        "Rural": (2, 0.5),
        sink: (3, 1.5),
    }

    fig, ax = plt.subplots(figsize=(12, 7))

    nx.draw_networkx(
        G,
        pos,
        ax=ax,
        node_size=2300,
        font_size=9,
        arrows=True,
        arrowsize=18,
    )

    labels = {
        (u, v): data["flow"]
        for u, v, data in G.edges(data=True)
    }

    nx.draw_networkx_edge_labels(
        G,
        pos,
        edge_labels=labels,
        font_size=8,
        ax=ax,
    )

    ax.set_title(
        "Optimal Survey Fieldwork Network"
    )

    ax.axis("off")

    fig.tight_layout()

    fig.savefig(
        FIGURES_DIR / "network_flow.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)


if __name__ == "__main__":
    plot_neyman()
    plot_allocation()
    plot_network()

    print("Figures saved in the figures directory.")

Figures saved in the figures directory.


In [7]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pulp


RESULTS_DIR = Path("results")
FIGURES_DIR = Path("figures")


def solve_with_capacity_factor(
    costs,
    capacities,
    targets,
    factor,
):
    teams = capacities["Team"].tolist()
    strata = targets["Stratum"].tolist()

    capacity = {
        row.Team: int(row.Capacity * factor)
        for row in capacities.itertuples()
    }

    demand = targets.set_index("Stratum")[
        "Neyman_Sample"
    ].to_dict()

    unit_cost = {
        (row.Team, row.Stratum): row.Cost
        for row in costs.itertuples()
    }

    model = pulp.LpProblem(
        "Capacity_Sensitivity",
        pulp.LpMinimize,
    )

    x = pulp.LpVariable.dicts(
        "x",
        (teams, strata),
        lowBound=0,
        cat=pulp.LpInteger,
    )

    model += pulp.lpSum(
        unit_cost[t, s] * x[t][s]
        for t in teams
        for s in strata
    )

    for s in strata:
        model += (
            pulp.lpSum(x[t][s] for t in teams)
            == demand[s]
        )

    for t in teams:
        model += (
            pulp.lpSum(x[t][s] for s in strata)
            <= capacity[t]
        )

    model.solve(
        pulp.PULP_CBC_CMD(msg=False)
    )

    status = pulp.LpStatus[model.status]

    if status != "Optimal":
        return None

    return pulp.value(model.objective)


def main():
    costs = pd.read_csv(
        RESULTS_DIR / "cost_matrix.csv"
    )

    capacities = pd.read_csv(
        RESULTS_DIR / "team_capacity.csv"
    )

    targets = pd.read_csv(
        RESULTS_DIR / "neyman_allocation.csv"
    )

    factors = [
        1.00,
        1.05,
        1.10,
        1.15,
        1.20,
        1.30,
    ]

    rows = []

    for factor in factors:
        objective = solve_with_capacity_factor(
            costs,
            capacities,
            targets,
            factor,
        )

        rows.append(
            {
                "Capacity_Factor": factor,
                "Total_Capacity": int(
                    (
                        capacities["Capacity"]
                        * factor
                    ).astype(int).sum()
                ),
                "Minimum_Cost": objective,
                "Status": (
                    "Optimal"
                    if objective is not None
                    else "Infeasible"
                ),
            }
        )

    results = pd.DataFrame(rows)

    results.to_csv(
        RESULTS_DIR / "sensitivity_results.csv",
        index=False,
    )

    feasible = results.dropna(
        subset=["Minimum_Cost"]
    )

    fig, ax = plt.subplots(figsize=(7, 4.5))

    ax.plot(
        feasible["Total_Capacity"],
        feasible["Minimum_Cost"],
        marker="o",
    )

    ax.set_title(
        "Effect of Team Capacity on Minimum Cost"
    )
    ax.set_xlabel("Total Available Interview Capacity")
    ax.set_ylabel("Minimum Fieldwork Cost")
    ax.grid(alpha=0.3)

    fig.tight_layout()

    fig.savefig(
        FIGURES_DIR / "sensitivity_analysis.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)

    print(results.to_string(index=False))


if __name__ == "__main__":
    main()

 Capacity_Factor  Total_Capacity  Minimum_Cost  Status
            1.00             310       6053.83 Optimal
            1.05             324       6033.21 Optimal
            1.10             340       6017.09 Optimal
            1.15             355       6002.37 Optimal
            1.20             372       5986.38 Optimal
            1.30             402       5959.74 Optimal


In [8]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pulp


# ============================================================
# Project paths and experiment settings
# ============================================================

DATA_PATH = Path("data/synthetic_survey_dataset.csv")
RESULTS_DIR = Path("results")
FIGURES_DIR = Path("figures")

RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

TOTAL_SAMPLE = 300
MAX_DEVIATION = 0.25

# The earlier experiment showed that the important transition
# occurs between lambda = 0.5 and lambda = 1.0.
LAMBDA_VALUES = [
    0.00,
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.95,
    1.00,
    1.25,
    1.50,
    2.00,
]


# ============================================================
# Load project inputs
# ============================================================

def load_inputs():
    survey = pd.read_csv(DATA_PATH)

    costs = pd.read_csv(
        RESULTS_DIR / "cost_matrix.csv"
    )

    capacities = pd.read_csv(
        RESULTS_DIR / "team_capacity.csv"
    )

    targets = pd.read_csv(
        RESULTS_DIR / "neyman_allocation.csv"
    )

    return survey, costs, capacities, targets


# ============================================================
# Statistical calculations
# ============================================================

def get_stratum_statistics(survey):
    stats = (
        survey.groupby("Stratum")
        .agg(
            N_h=("Respondent_ID", "count"),
            S_h=("Monthly_Income", "std"),
        )
        .reset_index()
    )

    stats["S2_h"] = stats["S_h"] ** 2

    return stats


def stratified_variance(allocation, stats):
    """
    Variance of the stratified sample mean with finite
    population correction.

    Var(y_st) =
        sum_h W_h^2 * (1 - n_h/N_h) * S_h^2/n_h
    """

    population_size = stats["N_h"].sum()

    variance = 0.0

    for row in stats.itertuples():

        n_h = allocation[row.Stratum]
        N_h = row.N_h
        S2_h = row.S2_h

        if n_h <= 0:
            return np.inf

        W_h = N_h / population_size

        fpc = 1 - n_h / N_h

        variance += (
            W_h ** 2
            * fpc
            * S2_h
            / n_h
        )

    return variance


# ============================================================
# Flexible Neyman optimization model
# ============================================================

def solve_model(
    costs,
    capacities,
    targets,
    stats,
    penalty,
):

    teams = capacities["Team"].tolist()
    strata = targets["Stratum"].tolist()

    capacity = (
        capacities
        .set_index("Team")["Capacity"]
        .to_dict()
    )

    target = (
        targets
        .set_index("Stratum")["Neyman_Sample"]
        .to_dict()
    )

    unit_cost = {
        (row.Team, row.Stratum): row.Cost
        for row in costs.itertuples()
    }

    if sum(capacity.values()) < TOTAL_SAMPLE:
        raise ValueError(
            "Total team capacity is smaller than "
            "the required sample size."
        )

    model = pulp.LpProblem(
        f"Flexible_Neyman_lambda_{penalty}",
        pulp.LpMinimize,
    )

    # Interviews assigned from team t to stratum s
    x = pulp.LpVariable.dicts(
        "interviews",
        (teams, strata),
        lowBound=0,
        cat=pulp.LpInteger,
    )

    # Positive and negative deviations from Neyman targets
    d_plus = pulp.LpVariable.dicts(
        "over_target",
        strata,
        lowBound=0,
        cat=pulp.LpContinuous,
    )

    d_minus = pulp.LpVariable.dicts(
        "under_target",
        strata,
        lowBound=0,
        cat=pulp.LpContinuous,
    )

    fieldwork_cost = pulp.lpSum(
        unit_cost[t, s] * x[t][s]
        for t in teams
        for s in strata
    )

    total_deviation = pulp.lpSum(
        d_plus[s] + d_minus[s]
        for s in strata
    )

    # Objective:
    # operational cost + penalty for departing from Neyman
    model += (
        fieldwork_cost
        + penalty * total_deviation,
        "Total_Penalized_Cost",
    )

    # --------------------------------------------------------
    # Team capacities
    # --------------------------------------------------------

    for t in teams:

        model += (
            pulp.lpSum(
                x[t][s]
                for s in strata
            )
            <= capacity[t],
            f"Capacity_{t}",
        )

    # --------------------------------------------------------
    # Flexible Neyman allocation
    # --------------------------------------------------------

    for s in strata:

        model += (
            pulp.lpSum(
                x[t][s]
                for t in teams
            )
            ==
            target[s]
            + d_plus[s]
            - d_minus[s],
            f"Neyman_Balance_{s}",
        )

    # --------------------------------------------------------
    # Keep total sample size fixed
    # --------------------------------------------------------

    model += (
        pulp.lpSum(
            x[t][s]
            for t in teams
            for s in strata
        )
        == TOTAL_SAMPLE,
        "Total_Sample_Size",
    )

    # --------------------------------------------------------
    # Prevent extreme departures from Neyman allocation
    # --------------------------------------------------------

    for s in strata:

        lower_bound = max(
            1,
            int(
                np.floor(
                    (1 - MAX_DEVIATION)
                    * target[s]
                )
            ),
        )

        upper_bound = int(
            np.ceil(
                (1 + MAX_DEVIATION)
                * target[s]
            )
        )

        realized = pulp.lpSum(
            x[t][s]
            for t in teams
        )

        model += (
            realized >= lower_bound,
            f"Lower_Bound_{s}",
        )

        model += (
            realized <= upper_bound,
            f"Upper_Bound_{s}",
        )

    # --------------------------------------------------------
    # Solve
    # --------------------------------------------------------

    solver = pulp.PULP_CBC_CMD(msg=False)

    model.solve(solver)

    status = pulp.LpStatus[model.status]

    if status != "Optimal":
        raise RuntimeError(
            f"No optimal solution for lambda={penalty}. "
            f"Solver status: {status}"
        )

    # --------------------------------------------------------
    # Extract solution
    # --------------------------------------------------------

    realized_allocation = {}
    allocation_rows = []
    team_rows = []

    for s in strata:

        realized = int(
            round(
                sum(
                    pulp.value(x[t][s])
                    for t in teams
                )
            )
        )

        realized_allocation[s] = realized

        allocation_rows.append(
            {
                "Lambda": penalty,
                "Stratum": s,
                "Neyman_Target": target[s],
                "Realized_Sample": realized,
                "Deviation": (
                    realized - target[s]
                ),
                "Absolute_Deviation": abs(
                    realized - target[s]
                ),
            }
        )

    # Store team-stratum flows as well.
    for t in teams:

        for s in strata:

            flow = int(
                round(
                    pulp.value(x[t][s])
                )
            )

            team_rows.append(
                {
                    "Lambda": penalty,
                    "Team": t,
                    "Stratum": s,
                    "Allocation": flow,
                    "Unit_Cost": unit_cost[t, s],
                    "Flow_Cost": (
                        flow * unit_cost[t, s]
                    ),
                }
            )

    actual_cost = pulp.value(fieldwork_cost)

    absolute_deviation = sum(
        abs(
            realized_allocation[s]
            - target[s]
        )
        for s in strata
    )

    estimator_variance = stratified_variance(
        realized_allocation,
        stats,
    )

    return {
        "lambda": penalty,
        "cost": actual_cost,
        "deviation": absolute_deviation,
        "variance": estimator_variance,
        "objective": pulp.value(model.objective),
        "allocation": allocation_rows,
        "team_allocation": team_rows,
    }


# ============================================================
# Plotting
# ============================================================

def plot_lambda_vs_cost(summary):

    fig, ax = plt.subplots(figsize=(7, 4.5))

    ax.plot(
        summary["Lambda"],
        summary["Fieldwork_Cost"],
        marker="o",
    )

    ax.set_xlabel("Penalty parameter (λ)")
    ax.set_ylabel("Minimum Fieldwork Cost")

    ax.set_title(
        "Fieldwork Cost as λ Increases"
    )

    ax.grid(alpha=0.3)

    fig.tight_layout()

    fig.savefig(
        FIGURES_DIR / "lambda_vs_cost.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)


def plot_lambda_vs_deviation(summary):

    fig, ax = plt.subplots(figsize=(7, 4.5))

    ax.plot(
        summary["Lambda"],
        summary["Total_Absolute_Deviation"],
        marker="o",
    )

    ax.set_xlabel("Penalty parameter (λ)")

    ax.set_ylabel(
        "Total Deviation from Neyman Allocation"
    )

    ax.set_title(
        "Convergence toward Neyman Allocation"
    )

    ax.grid(alpha=0.3)

    fig.tight_layout()

    fig.savefig(
        FIGURES_DIR / "lambda_vs_deviation.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)


def plot_lambda_vs_variance(summary):

    fig, ax = plt.subplots(figsize=(7, 4.5))

    ax.plot(
        summary["Lambda"],
        summary["Estimator_Variance"],
        marker="o",
    )

    ax.set_xlabel("Penalty parameter (λ)")

    ax.set_ylabel(
        "Variance of Stratified Mean Estimator"
    )

    ax.set_title(
        "Statistical Precision as λ Increases"
    )

    ax.grid(alpha=0.3)

    fig.tight_layout()

    fig.savefig(
        FIGURES_DIR / "lambda_vs_variance.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)


def plot_allocation_paths(allocations):

    pivot = allocations.pivot(
        index="Lambda",
        columns="Stratum",
        values="Realized_Sample",
    )

    fig, ax = plt.subplots(figsize=(8, 5))

    for stratum in pivot.columns:

        ax.plot(
            pivot.index,
            pivot[stratum],
            marker="o",
            label=stratum,
        )

    ax.set_xlabel("Penalty parameter (λ)")
    ax.set_ylabel("Realized Sample Size")

    ax.set_title(
        "Stratum Allocations as λ Changes"
    )

    ax.legend(title="Stratum")

    ax.grid(alpha=0.3)

    fig.tight_layout()

    fig.savefig(
        FIGURES_DIR / "lambda_allocation.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)


def plot_cost_variance_frontier(summary):
    """
    Collapse lambda values that produce the same
    cost-variance solution before adding labels.
    """

    frontier = (
        summary[
            [
                "Estimator_Variance",
                "Fieldwork_Cost",
                "Lambda",
            ]
        ]
        .groupby(
            [
                "Estimator_Variance",
                "Fieldwork_Cost",
            ],
            as_index=False,
        )
        .agg(
            Lambda_Min=("Lambda", "min"),
            Lambda_Max=("Lambda", "max"),
        )
        .sort_values("Estimator_Variance")
    )

    fig, ax = plt.subplots(figsize=(7, 4.5))

    ax.plot(
        frontier["Estimator_Variance"],
        frontier["Fieldwork_Cost"],
        marker="o",
    )

    for row in frontier.itertuples():

        if np.isclose(
            row.Lambda_Min,
            row.Lambda_Max,
        ):
            label = (
                f"λ={row.Lambda_Min:g}"
            )

        else:
            label = (
                f"λ={row.Lambda_Min:g}"
                f"–{row.Lambda_Max:g}"
            )

        ax.annotate(
            label,
            (
                row.Estimator_Variance,
                row.Fieldwork_Cost,
            ),
            xytext=(7, 7),
            textcoords="offset points",
            fontsize=8,
        )

    ax.set_xlabel(
        "Variance of Stratified Mean Estimator"
    )

    ax.set_ylabel("Fieldwork Cost")

    ax.set_title(
        "Cost–Precision Trade-off"
    )

    ax.grid(alpha=0.3)

    fig.tight_layout()

    fig.savefig(
        FIGURES_DIR
        / "cost_variance_frontier.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)

    return frontier


# ============================================================
# Main experiment
# ============================================================

def main():

    survey, costs, capacities, targets = (
        load_inputs()
    )

    stats = get_stratum_statistics(survey)

    neyman_target = (
        targets
        .set_index("Stratum")[
            "Neyman_Sample"
        ]
        .to_dict()
    )

    # Benchmark variance under exact Neyman allocation
    neyman_variance = stratified_variance(
        neyman_target,
        stats,
    )

    summary_rows = []
    allocation_rows = []
    team_allocation_rows = []

    for penalty in LAMBDA_VALUES:

        result = solve_model(
            costs,
            capacities,
            targets,
            stats,
            penalty,
        )

        variance_increase = (
            100
            * (
                result["variance"]
                - neyman_variance
            )
            / neyman_variance
        )

        relative_efficiency = (
            neyman_variance
            / result["variance"]
        )

        summary_rows.append(
            {
                "Lambda": penalty,

                "Fieldwork_Cost": round(
                    result["cost"],
                    2,
                ),

                "Total_Absolute_Deviation":
                    result["deviation"],

                "Estimator_Variance":
                    result["variance"],

                "Variance_Increase_Percent":
                    variance_increase,

                "Relative_Efficiency":
                    relative_efficiency,

                "Penalized_Objective":
                    result["objective"],
            }
        )

        allocation_rows.extend(
            result["allocation"]
        )

        team_allocation_rows.extend(
            result["team_allocation"]
        )

    summary = pd.DataFrame(summary_rows)

    allocations = pd.DataFrame(
        allocation_rows
    )

    team_allocations = pd.DataFrame(
        team_allocation_rows
    )

    # --------------------------------------------------------
    # Find first lambda giving exact Neyman allocation
    # --------------------------------------------------------

    exact_rows = summary[
        summary["Total_Absolute_Deviation"] == 0
    ]

    if not exact_rows.empty:

        threshold_lambda = exact_rows[
            "Lambda"
        ].min()

    else:
        threshold_lambda = None

    # --------------------------------------------------------
    # Save numerical results
    # --------------------------------------------------------

    summary.to_csv(
        RESULTS_DIR / "lambda_sensitivity.csv",
        index=False,
    )

    allocations.to_csv(
        RESULTS_DIR / "lambda_allocations.csv",
        index=False,
    )

    team_allocations.to_csv(
        RESULTS_DIR
        / "lambda_team_allocations.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Generate plots
    # --------------------------------------------------------

    plot_lambda_vs_cost(summary)

    plot_lambda_vs_deviation(summary)

    plot_lambda_vs_variance(summary)

    plot_allocation_paths(allocations)

    frontier = plot_cost_variance_frontier(
        summary
    )

    frontier.to_csv(
        RESULTS_DIR
        / "cost_precision_frontier.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Console report
    # --------------------------------------------------------

    print("\nNEyman benchmark")
    print("----------------")
    print(
        f"Target allocation: {neyman_target}"
    )
    print(
        f"Benchmark variance: "
        f"{neyman_variance:,.4f}"
    )

    print("\nLambda sensitivity")
    print("------------------")

    columns = [
        "Lambda",
        "Fieldwork_Cost",
        "Total_Absolute_Deviation",
        "Estimator_Variance",
        "Variance_Increase_Percent",
        "Relative_Efficiency",
    ]

    print(
        summary[columns].to_string(
            index=False,
            float_format=lambda x: f"{x:,.4f}",
        )
    )

    print("\nRealized stratum allocations")
    print("----------------------------")

    allocation_table = allocations.pivot(
        index="Lambda",
        columns="Stratum",
        values="Realized_Sample",
    )

    print(allocation_table.to_string())

    if threshold_lambda is not None:

        print(
            "\nFirst λ producing exact "
            f"Neyman allocation: "
            f"{threshold_lambda:g}"
        )

    else:

        print(
            "\nExact Neyman allocation was not "
            "reached over the tested λ range."
        )

    print(
        "\nResults saved to the results/ "
        "directory."
    )

    print(
        "Figures saved to the figures/ "
        "directory."
    )


if __name__ == "__main__":
    main()


NEyman benchmark
----------------
Target allocation: {'Rural': 59, 'Semi-Urban': 56, 'Urban': 185}
Benchmark variance: 752,660.9363

Lambda sensitivity
------------------
 Lambda  Fieldwork_Cost  Total_Absolute_Deviation  Estimator_Variance  Variance_Increase_Percent  Relative_Efficiency
 0.0000      6,031.2400                        30        783,085.8482                     4.0423               0.9611
 0.1000      6,031.2400                        30        783,085.8482                     4.0423               0.9611
 0.2000      6,031.2400                        30        783,085.8482                     4.0423               0.9611
 0.3000      6,031.2400                        30        783,085.8482                     4.0423               0.9611
 0.4000      6,031.2400                        30        783,085.8482                     4.0423               0.9611
 0.5000      6,031.2400                        30        783,085.8482                     4.0423               0.9611
 0